In [1]:
import os
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import rasterio
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterio.enums import Resampling as ResamplingEnum
from rasterio.transform import array_bounds
from rasterio.plot import plotting_extent
import osmnx as ox
from shapely.geometry import box

# -----------------------------
# User settings (edit as needed)
# -----------------------------
OUT_FIG = "figure1_transport_network.png"
DPI = 400

# Study area: Kinshasa (approx) bounding box in WGS84
# You SHOULD replace with your exact planning polygon if you have it.
NORTH, SOUTH, EAST, WEST = -4.10, -4.65, 15.45, 15.15

# Projection (as in many Kinshasa studies): WGS84 / UTM 33S
TARGET_CRS = "EPSG:32733"

# Input files
DEM_TIF = "data/dem.tif"
NE_COUNTRIES_SHP = "data/ne_10m_admin_0_countries.shp"  # optional inset

# Road classification
MAJOR_HWY = {"motorway", "trunk", "primary"}
ARTERIAL_HWY = {"secondary", "tertiary"}

# -----------------------------
# Helpers
# -----------------------------
def hillshade(dem, azimuth=315, altitude=45):
    """
    Simple hillshade from DEM array.
    """
    # gradients
    x, y = np.gradient(dem.astype("float64"))
    slope = np.pi/2.0 - np.arctan(np.hypot(x, y))
    aspect = np.arctan2(-x, y)

    az = np.deg2rad(azimuth)
    alt = np.deg2rad(altitude)

    shaded = np.sin(alt)*np.sin(slope) + np.cos(alt)*np.cos(slope)*np.cos(az - aspect)
    return np.clip(shaded, 0, 1)

def reproject_raster_to_crs(src_path, dst_crs):
    """
    Loads a raster and reprojects it in-memory to dst_crs.
    Returns (array, transform, crs).
    """
    with rasterio.open(src_path) as src:
        transform, width, height = calculate_default_transform(
            src.crs, dst_crs, src.width, src.height, *src.bounds
        )
        kwargs = src.meta.copy()
        kwargs.update({
            "crs": dst_crs,
            "transform": transform,
            "width": width,
            "height": height
        })

        dst = np.zeros((height, width), dtype=np.float32)

        reproject(
            source=rasterio.band(src, 1),
            destination=dst,
            src_transform=src.transform,
            src_crs=src.crs,
            dst_transform=transform,
            dst_crs=dst_crs,
            resampling=ResamplingEnum.bilinear
        )

    return dst, transform, dst_crs

def classify_osm_highway(val):
    """
    OSM 'highway' can be a string or a list. Return a single class label.
    """
    if isinstance(val, list) and len(val) > 0:
        val = val[0]
    if not isinstance(val, str):
        return "other"
    v = val.lower()
    if v in MAJOR_HWY:
        return "major"
    if v in ARTERIAL_HWY:
        return "arterial"
    return "other"

# -----------------------------
# Main
# -----------------------------
def main():
    os.makedirs(os.path.dirname(OUT_FIG) or ".", exist_ok=True)

    # --- 1) Define study area polygon (WGS84)
    study_poly_wgs84 = box(WEST, SOUTH, EAST, NORTH)

    # --- 2) Download OSM data (roads + rail) within polygon
    # This uses Overpass; you need internet for this step.
    ox.settings.use_cache = True
    ox.settings.log_console = False

    # Roads
    road_tags = {"highway": True}
    roads = ox.features_from_polygon(study_poly_wgs84, tags=road_tags)
    roads = roads[roads.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    roads = roads.to_crs(TARGET_CRS)

    # Rail
    rail_tags = {"railway": True}
    rails = ox.features_from_polygon(study_poly_wgs84, tags=rail_tags)
    rails = rails[rails.geometry.type.isin(["LineString", "MultiLineString"])].copy()
    rails = rails.to_crs(TARGET_CRS)

    # --- 3) Classify roads
    if "highway" not in roads.columns:
        roads["highway"] = None
    roads["road_class"] = roads["highway"].apply(classify_osm_highway)

    roads_major = roads[roads["road_class"] == "major"]
    roads_art = roads[roads["road_class"] == "arterial"]
    roads_other = roads[roads["road_class"] == "other"]

    # --- 4) DEM processing (relief + hillshade)
    if not os.path.exists(DEM_TIF):
        raise FileNotFoundError(
            f"DEM not found at {DEM_TIF}. Please download a DEM GeoTIFF and place it there."
        )

    dem, dem_transform, dem_crs = reproject_raster_to_crs(DEM_TIF, TARGET_CRS)

    # Mask unrealistic no-data if present (common: very negative)
    dem = np.where(dem < -1000, np.nan, dem)

    hs = hillshade(np.nan_to_num(dem, nan=np.nanmin(dem)))

    dem_extent = plotting_extent(dem, dem_transform)

    # --- 5) Create figure layout (main map + inset)
    fig = plt.figure(figsize=(12, 9))
    ax = fig.add_axes([0.05, 0.08, 0.72, 0.85])   # main map
    ax_in = fig.add_axes([0.79, 0.62, 0.19, 0.31])  # inset (top-right)
    ax_leg = fig.add_axes([0.79, 0.08, 0.19, 0.50]) # legend/notes
    ax_leg.axis("off")

    # --- 6) Plot relief (own styling)
    # elevation colour (no copyrighted palette; use a standard matplotlib colormap)
    ax.imshow(dem, extent=dem_extent, origin="upper", alpha=0.95)
    # hillshade overlay (multiply-like effect by alpha)
    ax.imshow(hs, extent=dem_extent, origin="upper", alpha=0.35, cmap="gray")

    # --- 7) Plot network
    # Choose your own line widths; do not replicate JICA styling exactly.
    if len(roads_other) > 0:
        roads_other.plot(ax=ax, linewidth=0.4, alpha=0.7)
    if len(roads_art) > 0:
        roads_art.plot(ax=ax, linewidth=1.0, alpha=0.9)
    if len(roads_major) > 0:
        roads_major.plot(ax=ax, linewidth=1.8, alpha=0.95)

    if len(rails) > 0:
        rails.plot(ax=ax, linewidth=1.0, linestyle="--", alpha=0.8)

    # --- 8) Set main extent to study area
    study_poly = gpd.GeoSeries([study_poly_wgs84], crs="EPSG:4326").to_crs(TARGET_CRS).iloc[0]
    minx, miny, maxx, maxy = study_poly.bounds
    ax.set_xlim(minx, maxx)
    ax.set_ylim(miny, maxy)

    ax.set_title("Transport Network and Elevation – Kinshasa Study Area", fontsize=14)
    ax.set_xlabel("Easting (m) – UTM 33S")
    ax.set_ylabel("Northing (m) – UTM 33S")

    # --- 9) Inset map: highlight DRC and study area
    try:
        if os.path.exists(NE_COUNTRIES_SHP):
            countries = gpd.read_file(NE_COUNTRIES_SHP)
            # Filter DRC by common names; adjust if needed
            drc = countries[countries["ADMIN"].str.contains("Democratic Republic", case=False, na=False)].copy()
            if len(drc) == 0:
                drc = countries[countries["NAME"].str.contains("Congo", case=False, na=False)].copy()

            drc = drc.to_crs("EPSG:4326")
            drc.plot(ax=ax_in, linewidth=0.5, edgecolor="black", facecolor="none")
            # Study polygon box
            gpd.GeoSeries([study_poly_wgs84], crs="EPSG:4326").plot(
                ax=ax_in, edgecolor="red", facecolor="none", linewidth=1.5
            )
            ax_in.set_title("Inset: DRC & Study Area", fontsize=10)
            ax_in.set_axis_off()
        else:
            ax_in.text(0.5, 0.5, "Inset skipped:\nNatural Earth not found", ha="center", va="center")
            ax_in.set_axis_off()
    except Exception as e:
        ax_in.text(0.5, 0.5, f"Inset error:\n{e}", ha="center", va="center")
        ax_in.set_axis_off()

    # --- 10) Legend (custom, original)
    major_line = mlines.Line2D([], [], linewidth=2.5, label="Major arterial (OSM primary/trunk/motorway)")
    art_line = mlines.Line2D([], [], linewidth=1.5, label="Arterial (OSM secondary/tertiary)")
    other_line = mlines.Line2D([], [], linewidth=0.8, label="Other roads (OSM other highway)")
    rail_line = mlines.Line2D([], [], linestyle="--", linewidth=1.5, label="Railway (OSM)")

    ax_leg.legend(handles=[major_line, art_line, other_line, rail_line], loc="upper left", frameon=False)

    # Attribution note (important for editor)
    note = (
        "Data & licensing notes (for caption):\n"
        "• Roads/Rail: © OpenStreetMap contributors (ODbL)\n"
        "• Elevation: DEM (e.g., SRTM/Copernicus) with attribution\n"
        "• Boundaries (inset): Natural Earth (public domain)\n"
        "Cartography and styling: Authors.\n"
    )
    ax_leg.text(0.0, 0.05, note, fontsize=9, va="bottom")

    # --- 11) Export
    plt.savefig(OUT_FIG, dpi=DPI, bbox_inches="tight")
    plt.close(fig)

    print(f"Saved: {OUT_FIG}")

if __name__ == "__main__":
    main()

ModuleNotFoundError: No module named 'geopandas'